### Import the required libraries

Import `yfinance` to download historical stock prices, `pandas` to transform the data, and `psycopg2` to load the results into PostgreSQL.

In [1]:
import yfinance as yf
import pandas as pd
import psycopg2

### Download daily Apple stock prices

Download daily data for the `AAPL` symbol using a start date of `2023-01-01` and an exclusive end date of `2025-01-01`. Move the date index into a column and display the first five rows to inspect the downloaded data.

In [2]:
symbol = "AAPL"
df = yf.download(
    symbol,
    start="2023-01-01",
    end="2025-01-01",
    interval="1d"
)

df = df.reset_index()
df.head()

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open,Volume
Ticker,,AAPL,AAPL,AAPL,AAPL,AAPL
0,2023-01-03,122.876747,128.604505,121.992528,127.995383,112117500
1,2023-01-04,124.144119,126.403789,122.886567,124.664824,89113600
2,2023-01-05,122.827614,125.529389,122.572179,124.900613,80962700
3,2023-01-06,127.346947,128.005196,122.699897,123.800259,87754700
4,2023-01-09,127.867676,131.070518,127.612241,128.182071,70790800


### Flatten column names and reset the index

If the download produced multi-level column names, keep their first level so columns such as `Close` and `Volume` can be accessed directly. Reset the row index again; because the previous cell already reset it, this also retains the old row index as a column.

In [3]:
import pandas as pd

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df.reset_index()

### Inspect the volume value type

Print the Python type of the first row's `Volume` value to check how pandas represents it before preparing the data for PostgreSQL.

In [4]:
print(type(df.loc[0, "Volume"]))

<class 'numpy.int64'>


### Calculate daily returns and rolling volatility

Calculate the fractional change in closing price from one trading row to the next and store it in `return`. Calculate the rolling standard deviation of those returns over 20 trading rows as `volatility_20d`. Remove rows containing missing values, including the initial rows without enough history, and preview the results.

In [5]:
#transform
df["return"] = df["Close"].pct_change()

# rolling volatility 20 hari
df["volatility_20d"] = df["return"].rolling(20).std()

df = df.dropna()
df.head()

Price,index,Date,Close,High,Low,Open,Volume,return,volatility_20d
20,20,2023-02-01,142.879700,144.039015,138.841788,141.445311,77663600,0.007901,0.012758
21,21,2023-02-02,148.175201,148.528874,145.571663,146.288858,118339000,0.037063,0.014354
22,22,2023-02-03,151.790649,154.620150,145.237618,145.434108,154357300,0.024400,0.013969
23,23,2023-02-06,149.069229,150.415215,148.135892,149.894510,69858300,-0.017929,0.013955
24,24,2023-02-07,151.937988,152.507819,147.998315,147.998315,83322600,0.019244,0.014142


### Prepare the index and numeric types

Reset the filtered DataFrame to a consecutive row index without keeping the old index. Convert `Volume` to integers and the return and volatility columns to floating-point values for the database load.

In [6]:
df = df.reset_index(drop=True)

df["Volume"] = df["Volume"].astype(int)
df["return"] = df["return"].astype(float)
df["volatility_20d"] = df["volatility_20d"].astype(float)

### Inspect the prepared DataFrame

Display the DataFrame's column names, non-null counts, data types, and memory usage to review its structure before inserting rows into PostgreSQL.

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 482 entries, 0 to 481
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype        
---  ------          --------------  -----        
 0   index           482 non-null    int64        
 1   Date            482 non-null    datetime64[s]
 2   Close           482 non-null    float64      
 3   High            482 non-null    float64      
 4   Low             482 non-null    float64      
 5   Open            482 non-null    float64      
 6   Volume          482 non-null    int64        
 7   return          482 non-null    float64      
 8   volatility_20d  482 non-null    float64      
dtypes: datetime64[s](1), float64(6), int64(2)
memory usage: 34.0 KB


### Connect to PostgreSQL and create a cursor

Connect to the local `streaming_db` database using the configured credentials and create a cursor for inserting the prepared stock data.

In [8]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="streaming_db",
    user="postgres",
    password=""
)
cur = conn.cursor()

### Insert historical stock data and close the connection

Insert each prepared row into the existing `yahoo_finance.stock_price` table, including the symbol, date, OHLC prices, volume, daily return, and 20-row volatility. Convert volume, return, and volatility values to native Python numeric types when binding SQL parameters. Commit all inserts, close the cursor and connection, and print a completion message.

In [9]:
for _, row in df.iterrows():
    cur.execute("""
        INSERT INTO yahoo_finance.stock_price
        (symbol, date, open, high, low, close, volume, return, volatility_20d)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, (
        symbol,
        row["Date"],
        row["Open"],
        row["High"],
        row["Low"],
        row["Close"],
        int(row["Volume"]),
        float(row["return"]),
        float(row["volatility_20d"])
    ))

conn.commit()
cur.close()
conn.close()

print("✅ Historical data loaded to Postgres")

✅ Historical data loaded to Postgres
